# A4. Negative Review Batch Analysis Notebook

> **Companion module**: [A4 Customer Service & After-Sales](../paths/a-operators/a4-customer-service.md)
>
> **Function**: Upload negative review CSV → Auto-classify + Frequency stats + Improvement suggestions + Visualization
>
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kangise/ecommerce-ai-roadmap/blob/main/notebooks/a4-negative-review-analysis.ipynb)

---

## 1. Install Dependencies

In [ ]:
!pip install -q pandas numpy plotly wordcloud matplotlib

## 2. Load Negative Review Data

In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter

# === Option 1: Upload your CSV ===
# from google.colab import files
# uploaded = files.upload()
# df = pd.read_csv(list(uploaded.keys())[0])

# === Option 2: Sample data (100 negative reviews) ===
np.random.seed(42)
neg_titles = [
    'Battery dies in 2 hours', 'Bluetooth keeps disconnecting', 'Broke after one week',
    'Terrible sound quality', 'Does not fit my ears', 'Charging case stopped working',
    'Not noise cancelling at all', 'Left earbud stopped working', 'Uncomfortable after 30 min',
    'Worse than my old earbuds', 'Returned immediately', 'False advertising on battery life',
    'Connection drops every 5 minutes', 'Cheap plastic build', 'One side louder than other'
]
neg_bodies = [
    'Battery only lasts 2 hours, not 8 as advertised. Complete waste of money. Had to return.',
    'The bluetooth connection drops every few minutes. Tried with iPhone and Android, same issue.',
    'The charging case lid broke after one week. Cheap plastic construction. Very disappointed.',
    'Sound quality is tinny with no bass at all. My $15 earbuds sound better than these.',
    'Tried all 4 ear tip sizes but none fit properly. They keep falling out during exercise.',
    'Left earbud completely stopped working after 3 weeks. No response from customer service.',
    'The noise cancellation is basically non-existent. Can still hear everything around me.',
    'After 30 minutes of wearing, my ears start hurting. The design is not ergonomic at all.',
    'Charging case takes forever to charge and the LED indicator is confusing.',
    'Audio delay when watching videos. Lip sync is off by about half a second.'
]

n = 100
df = pd.DataFrame({
    'rating': np.random.choice([1, 2], n, p=[0.4, 0.6]),
    'title': [neg_titles[i % len(neg_titles)] for i in range(n)],
    'body': [neg_bodies[i % len(neg_bodies)] for i in range(n)],
    'date': pd.date_range('2025-06-01', periods=n, freq='2D'),
    'helpful_votes': np.random.poisson(3, n)
})

df['full_text'] = df['title'] + '. ' + df['body']
print(f'Loaded {len(df)} negative reviews (1-2 stars)')
print(f'Rating distribution: {df["rating"].value_counts().to_dict()}')
df.head()

## 3. Keyword Extraction & Issue Classification

In [ ]:
# Define issue categories and keywords
ISSUE_CATEGORIES = {
    '🔋 Battery Life': ['battery', 'charge', 'dies', 'hours', 'drain', 'last'],
    '📶 Bluetooth Connection': ['bluetooth', 'connection', 'disconnect', 'drops', 'pair', 'connect'],
    '🔨 Build Quality': ['broke', 'broken', 'cheap', 'plastic', 'build', 'quality', 'fell apart'],
    '🔊 Sound Quality': ['sound', 'audio', 'bass', 'tinny', 'volume', 'louder', 'delay'],
    '👂 Wearing Comfort': ['fit', 'comfortable', 'ear', 'pain', 'fall', 'size', 'tip'],
    '🔇 Noise Cancellation': ['noise', 'cancellation', 'anc', 'cancel'],
    '📦 Charging Case': ['case', 'charging case', 'lid', 'led', 'indicator'],
    '🛠️ Functional Defect': ['stopped working', 'defective', 'malfunction', 'dead']
}

def classify_issue(text):
    text_lower = text.lower()
    matches = []
    for category, keywords in ISSUE_CATEGORIES.items():
        if any(kw in text_lower for kw in keywords):
            matches.append(category)
    return matches if matches else ['❓ Other']

df['issues'] = df['full_text'].apply(classify_issue)
df['primary_issue'] = df['issues'].apply(lambda x: x[0])

# Statistics
issue_counts = df.explode('issues')['issues'].value_counts()
print('=== Negative Review Issue Classification Ranking ===')
for issue, count in issue_counts.items():
    pct = count / len(df) * 100
    bar = '█' * int(pct / 2)
    print(f'{issue}: {count} ({pct:.0f}%) {bar}')

## 4. Visualization

In [ ]:
import plotly.express as px
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Issue distribution pie chart
fig = px.pie(values=issue_counts.values, names=issue_counts.index,
             title='Negative Review Issue Distribution', hole=0.3)
fig.show()

# Issue trend (monthly)
df['month'] = pd.to_datetime(df['date']).dt.to_period('M').astype(str)
exploded = df.explode('issues')
monthly_issues = exploded.groupby(['month', 'issues']).size().reset_index(name='count')
fig = px.line(monthly_issues, x='month', y='count', color='issues',
              title='Monthly Negative Review Issue Trend (Which issues are getting worse?)')
fig.show()

# Word cloud
all_text = ' '.join(df['full_text'].tolist())
wc = WordCloud(width=800, height=400, background_color='white', max_words=60, colormap='Reds').generate(all_text)
plt.figure(figsize=(12, 6))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Negative Review Keyword Word Cloud', fontsize=16)
plt.show()

## 5. Generate Improvement Suggestions

In [ ]:
print('=== Product Improvement Priority Suggestions ===')
print()
for i, (issue, count) in enumerate(issue_counts.head(5).items(), 1):
    pct = count / len(df) * 100
    severity = '🔴 Urgent' if pct > 20 else ('🟡 Important' if pct > 10 else '🟢 Monitor')
    print(f'{i}. {issue} — {count} negative reviews ({pct:.0f}%) [{severity}]')
    # Show typical negative reviews for this category
    examples = df[df['primary_issue'] == issue]['full_text'].head(2)
    for ex in examples:
        print(f'   → "{ex[:80]}..."')
    print()

print('\n=== Action Items ===')
print('P0 (This week): Fix the #1 issue — directly impacts return rate and ratings')
print('P1 (This month): Address issues #2-3 — manage expectations in the listing')
print('P2 (Next month): Pre-populate Q&A with answers for the top 5 issues (Rufus optimization)')

## 6. Export

In [ ]:
# Export classified negative review data
df[['date', 'rating', 'title', 'body', 'primary_issue', 'helpful_votes']].to_csv('negative_reviews_classified.csv', index=False)

# Export issue summary
summary = issue_counts.reset_index()
summary.columns = ['Issue', 'Count']
summary['Percentage'] = (summary['Count'] / len(df) * 100).round(1)
summary.to_csv('issue_summary.csv', index=False)

print('Exported: negative_reviews_classified.csv, issue_summary.csv')